<a href="https://colab.research.google.com/github/Pmskabir1234/Torch/blob/main/Dataset_DataLoader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In PyTorch, Dataset and DataLoader are fundamental for efficient data management and loading, especially when dealing with large datasets.

###**Dataset:**

**Why it's important:**

The Dataset class in PyTorch provides an abstract interface for accessing data. It defines three core methods:

**__init__(self,features,labels)**: The contructor to load the dataset.

**__len__(self):** Returns the total number of samples in the dataset.

**__getitem__(self, idx):** Retrieves a single sample (e.g., an image and its label) given an index idx. If we want transform the dataset, we do it here(such as resizing, lemmatizing etc).

How it works: You create a custom Dataset class that inherits from torch.utils.data.Dataset. In this class, you specify how to load, preprocess, and transform individual data samples from your raw data source (e.g., files on disk, a database). This abstraction decouples data loading and preprocessing logic from your model training loop, making your code cleaner and more modular.


###**DataLoader:**

**Why it's important:** The DataLoader class wraps a Dataset and provides an iterable over it. It's crucial for efficiently feeding data to your model during training and evaluation.

**How it works:** It handles several key aspects:
Batching: It groups individual samples from the Dataset into mini-batches, which are essential for stochastic gradient descent (SGD) and its variants.

**Shuffling:** It can shuffle the data at the beginning of each epoch to prevent the model from memorizing the order of samples, improving generalization.

**Parallelism (Multiprocessing):** It can use multiple worker processes *(num_workers > 0)* to load data in parallel in the background while your GPU is busy training. This prevents data loading from becoming a bottleneck, significantly speeding up training for large datasets.

**Memory Management:** It loads data in batches, which is memory-efficient as it doesn't need to load the entire dataset into RAM at once.

**drop_last** is used if the total number of rows are not divisible by batches and we want to drop the last incomplete  batch.

In [4]:
import torch
from torch.utils.data import Dataset,DataLoader
import numpy as np

In [15]:
X = np.random.randint(10,30,size=(10,10))
y = np.random.randint(0,2,10)

Created a dummy data of features shape (10,10) and label shape (10,10). Now we will see the use & implementation of Dataset and DataLoader.

In [34]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features).float()
    self.labels = torch.tensor(labels).long()

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [35]:
data = CustomDataset(X,y)

In [36]:
data[1] #any row can be accessed now by index, use of getitem from CustomDataset


(tensor([16., 13., 22., 25., 22., 13., 26., 15., 17., 14.]), tensor(1))

In [44]:
loader = DataLoader(data, batch_size = 3, drop_last = True, num_workers=2, shuffle=True)

In [45]:
for features,labels in loader:
  print(features)
  print(labels)
  print("-"*30)

tensor([[16., 13., 22., 25., 22., 13., 26., 15., 17., 14.],
        [20., 29., 12., 11., 18., 22., 12., 18., 12., 22.],
        [19., 22., 21., 18., 20., 23., 17., 21., 11., 15.]])
tensor([1, 0, 1])
------------------------------
tensor([[12., 16., 16., 25., 18., 15., 11., 10., 28., 15.],
        [20., 17., 17., 18., 23., 26., 21., 28., 28., 21.],
        [27., 18., 20., 17., 26., 18., 29., 21., 23., 25.]])
tensor([1, 1, 0])
------------------------------
tensor([[12., 14., 16., 22., 11., 25., 19., 15., 19., 15.],
        [13., 15., 28., 14., 17., 27., 24., 12., 16., 26.],
        [14., 14., 22., 27., 13., 23., 17., 27., 26., 13.]])
tensor([1, 1, 1])
------------------------------


##Observation

1. The Dataloader returns a iterable which elements can be accessed using loop. Since we returned both features and labels in __getitem__, we can access both in loader.

2. num_workers helps in parallelism, helps to form batches from chunks faster.

###Workflow
Dataset loads data **->** Loader shuffles rows by index **->** batch_size creates groups shuffled indices **->** getitem method used to get rows from indices to form data batch **->** num_workers makes batch forming faster **->** bathces sent to pipeline for training.